# 矩阵乘法

本教程将编写一个简短的高性能 FP16 矩阵乘法内核，目标是达到与 cuBLAS 或 rocBLAS 相当的性能。

你将学习：

- 块级矩阵乘法。
- 多维指针运算。
- 通过调整程序实例的执行顺序提高 L2 缓存命中率。
- 自动性能调优。
---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 动机

矩阵乘法是现代高性能计算系统的重要基础操作，但优化难度很高，因此通常由硬件厂商在 cuBLAS 等内核库中提供实现。

这类库往往是专有的，难以针对现代深度学习工作负载定制，例如融合激活函数。本教程介绍如何使用 Triton 实现便于定制和扩展的高效矩阵乘法。

下面的代码框对应将形状为 (M, K) 和 (K, N) 的矩阵相乘的分块算法。

外层双重循环的每次迭代由一个独立的 Triton 程序实例完成。

## 计算内核

上述算法在 Triton 中的实现相对直接。主要难点是计算内层循环中 A 和 B 的数据块所在的内存地址，这需要多维指针运算。

### 指针运算

对于按行优先存储的二维张量 X，元素 $(i,j)$ 的地址等于基地址加上两个维度各自的偏移量：

$$
\operatorname{addr}(X_{i,j})=\operatorname{base}(X)+i\,s_i+j\,s_j
$$

其中 $s_i$ 和 $s_j$ 表示对应维度的步长。下面的代码框对应 A 和 B 子块的指针构造伪代码。

由此可以在 K 方向的起点初始化两个数据块的指针。如果 M 或 N 不是相应块大小的整数倍，需要额外的取模处理，以填入不会影响最终结果的值。K 方向的边界稍后通过带掩码的加载处理。

随后，在内层循环中推进指针。

### L2 缓存优化

每个程序实例计算输出矩阵 C 中一个大小为 BLOCK_SIZE_M × BLOCK_SIZE_N 的块。

块的计算顺序会影响 L2 缓存命中率。简单的行优先顺序不一定能带来理想效果。

一种改进方式是采用有利于数据复用的块启动顺序：先把若干行的块组成一个大组，再切换到下一列。

原文用每个矩阵由 9 × 9 个块组成的例子说明：计算前 9 个输出块时，行优先顺序需要向 SRAM 加载 90 个块，而分组顺序只需加载 54 个块。这些数字属于算法示例，并非本地测量结果。

原文还给出了顺序对比示意图及特定硬件上的性能结果，本笔记省略这些图表和测量数值。

## 完整实现

现在可以创建一个只接受两个输入张量的包装函数，依次检查形状约束、分配输出张量并启动上述内核。

## 单元测试

将自定义矩阵乘法与 PyTorch 的原生实现进行比较；原教程在这里使用 cuBLAS 作为参考。

## 性能基准测试

### 方阵性能

比较自定义内核与 cuBLAS 或 rocBLAS 的性能。这里主要测试方阵，也可以调整测试范围，研究其他矩阵形状。